# FungiGrowthJ — Biological Analysis Report

Generic notebook for any FungiGrowthJ `results` ZIP.

The workflow separates **central colony**, **satellite colonies**, and the **global colony phenotype**. It automatically discovers strains, biological replicates, dates, revisions, colony labels and available measurements. Strain comparisons use the biological replicate as the experimental unit; repeated observations through time are used for trajectories and rate estimation.

Satellite counts are interpreted as the number of **individually distinguishable satellite colonies**, not necessarily the number of biological colonies that exist. A decrease in distinguishable satellite count together with increasing satellite area is therefore treated as compatible with colony expansion, merging, or loss of separability—not automatically as satellite disappearance.


In [ ]:
from pathlib import Path

# USER CONFIGURATION
ZIP_PATH = Path("/mnt/data/results(1).zip")
OUTPUT_DIR = Path("FGJ_biological_report")

MAIN_COLONY_LABELS = {"CEN","CENTRAL","MAIN","PRIMARY","MAIN_COLONY","CENTRAL_COLONY"}
MULTIPLE_MAIN_POLICY = "exclude"   # rigorous default; alternatively "largest"
ALPHA = 0.05
MIN_POINTS_FOR_RATE = 3
SATELLITE_LABELS = {"SAT", "SATELLITE", "SATELLITES"}

OUTPUT_DIR.mkdir(exist_ok=True)
FIG_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"
FIG_DIR.mkdir(exist_ok=True)
TABLE_DIR.mkdir(exist_ok=True)

print("Input:", ZIP_PATH)


In [ ]:
import io
import re
import zipfile
import warnings
import base64
from datetime import datetime
from html import escape

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.cluster.hierarchy import linkage, dendrogram
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

warnings.filterwarnings("ignore")

def first_existing(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def slug(x):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(x))


In [ ]:
# LOAD RAW FungiGrowthJ RESULT FILES

if not ZIP_PATH.exists():
    zips = sorted(Path(".").glob("*.zip"))
    if len(zips) == 1:
        ZIP_PATH = zips[0]
        print("Using:", ZIP_PATH)
    else:
        raise FileNotFoundError(
            f"Could not find {ZIP_PATH}. Change ZIP_PATH in the configuration cell."
        )

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    result_files = [
        n for n in z.namelist()
        if not n.endswith("/")
        and n.lower().endswith(".csv")
        and "fungigrowthj_results" in Path(n).name.lower()
    ]

if not result_files:
    raise ValueError("No *_FungiGrowthJ_results.csv files found inside the ZIP.")

frames = []
with zipfile.ZipFile(ZIP_PATH, "r") as z:
    for name in result_files:
        try:
            d = pd.read_csv(io.BytesIO(z.read(name)))
            d["_source_path"] = name
            frames.append(d)
        except Exception as exc:
            print("Skipped:", name, "->", exc)

raw = pd.concat(frames, ignore_index=True, sort=False)
print("Raw CSV files:", len(result_files))
print("Raw rows:", len(raw))


In [ ]:
# NORMALIZE METADATA AND RESOLVE REVISIONS

def path_revision(path):
    m = re.search(r"revision[_-]?(\d+)", str(path), re.I)
    return int(m.group(1)) if m else 0

date_col = first_existing(raw, ["date","date_raw","observation_date"])
strain_col = first_existing(raw, ["strain_id","strain","isolate_id"])
rep_col = first_existing(raw, ["replicate_id","replicate","rep"])
image_col = first_existing(raw, ["image_file","image","filename"])

if not date_col:
    raise ValueError("No recognizable date column was found.")
if not strain_col or not rep_col:
    raise ValueError("Could not identify strain and replicate columns.")

raw["strain_id"] = raw[strain_col].astype(str)
raw["replicate_id"] = raw[rep_col].astype(str)
raw["date"] = pd.to_datetime(raw[date_col], errors="coerce").dt.normalize()

if raw["date"].isna().all():
    raise ValueError("Dates were found but could not be parsed.")

raw["image_file"] = (
    raw[image_col].astype(str)
    if image_col else raw["_source_path"].map(lambda x: Path(x).name)
)

analysis_rev = (
    pd.to_numeric(raw["analysis_revision"], errors="coerce").fillna(0).to_numpy()
    if "analysis_revision" in raw.columns
    else np.zeros(len(raw))
)
path_rev = raw["_source_path"].map(path_revision).to_numpy()
raw["_revision_rank"] = np.maximum(analysis_rev, path_rev)

raw["_image_key"] = (
    raw["strain_id"] + "||" +
    raw["replicate_id"] + "||" +
    raw["date"].astype(str) + "||" +
    raw["image_file"]
)

max_rev = raw.groupby("_image_key")["_revision_rank"].transform("max")
selected = raw.loc[raw["_revision_rank"] == max_rev].copy()

print("Rows after revision resolution:", len(selected))
print("Unique images:", selected["_image_key"].nunique())


In [ ]:
# DATA AUDIT

rep_counts = (
    selected[["strain_id","replicate_id"]]
    .drop_duplicates()
    .groupby("strain_id").size()
    .rename("n_replicates").reset_index()
)

date_counts = (
    selected.groupby(["strain_id","replicate_id"])["date"]
    .nunique().rename("n_dates").reset_index()
)

print("Strains:", selected["strain_id"].nunique())
print("Biological replicates:", len(selected[["strain_id","replicate_id"]].drop_duplicates()))
print("Dates:", selected["date"].nunique())

display(rep_counts)
display(date_counts)

if "final_type" in selected.columns:
    print("Final colony labels:")
    display(selected["final_type"].value_counts(dropna=False))

if "software_version" in selected.columns:
    print("Software version(s):")
    display(selected["software_version"].dropna().astype(str).unique())


In [ ]:
# IDENTIFY MAIN COLONY EXPLICITLY

def identify_main_label(df):
    found = []
    for col in ["final_type","proposed_type"]:
        if col not in df.columns:
            continue
        for value in df[col].dropna().astype(str).str.strip().unique():
            token = re.sub(r"[^A-Za-z0-9]+", "_", value.upper()).strip("_")
            if token in MAIN_COLONY_LABELS:
                found.append((col,value))
    found.sort(key=lambda x: 0 if x[0] == "final_type" else 1)
    return found[0] if found else None

main_info = identify_main_label(selected)
if main_info is None:
    raise ValueError(
        "No explicit main-colony label was recognized. "
        "The notebook will not silently use SAT/IGN as the main colony."
    )

main_col, main_label = main_info
selected["_main_candidate"] = (
    selected[main_col].astype(str).str.strip().str.upper()
    == str(main_label).strip().upper()
)

main_counts = selected.groupby("_image_key")["_main_candidate"].sum()

print(f"Main colony: {main_label!r} from {main_col!r}")
print("Images with 0 main candidates:", int((main_counts == 0).sum()))
print("Images with 1 main candidate:", int((main_counts == 1).sum()))
print("Images with >1 main candidate:", int((main_counts > 1).sum()))


In [ ]:
# BUILD DATE-LEVEL BIOLOGICAL OBSERVATIONS

candidate = selected[selected["_main_candidate"]].copy()

if "area_mm2" in candidate.columns:
    candidate["area_mm2"] = pd.to_numeric(candidate["area_mm2"], errors="coerce")

if MULTIPLE_MAIN_POLICY == "largest":
    candidate["_area_rank"] = candidate["area_mm2"].fillna(-np.inf)
    candidate = (
        candidate.sort_values(["_image_key","_area_rank"], ascending=[True,False])
        .drop_duplicates("_image_key")
    )
elif MULTIPLE_MAIN_POLICY == "exclude":
    counts = candidate.groupby("_image_key").size().rename("_n_main")
    candidate = candidate.merge(counts, on="_image_key", how="left")
    candidate = candidate[candidate["_n_main"] == 1].copy()
else:
    raise ValueError("MULTIPLE_MAIN_POLICY must be 'exclude' or 'largest'.")

numeric_vars = [
    "area_mm2","perimeter_mm","equivalent_radius_mm","equivalent_diameter_mm",
    "major_axis_mm","minor_axis_mm","feret_max_mm","feret_min_mm",
    "circularity","aspect_ratio","eccentricity","solidity","convexity",
    "plate_occupancy_pct"
]

for c in numeric_vars:
    if c in candidate.columns:
        candidate[c] = pd.to_numeric(candidate[c], errors="coerce")

agg = {c:"median" for c in numeric_vars if c in candidate.columns}
for c in ["experiment","software_version","scale_unit","scale_method"]:
    if c in candidate.columns:
        agg[c] = "first"

bio = candidate.groupby(
    ["strain_id","replicate_id","date"], as_index=False
).agg(agg)

n_images = (
    candidate.groupby(["strain_id","replicate_id","date"])["_image_key"]
    .nunique().rename("n_images").reset_index()
)
bio = bio.merge(n_images, on=["strain_id","replicate_id","date"], how="left")

if "equivalent_radius_mm" not in bio.columns:
    bio["equivalent_radius_mm"] = np.nan

bio["equivalent_radius_mm"] = pd.to_numeric(
    bio["equivalent_radius_mm"], errors="coerce"
)

derived = bio["equivalent_radius_mm"].isna() & bio["area_mm2"].notna()
bio.loc[derived,"equivalent_radius_mm"] = np.sqrt(
    bio.loc[derived,"area_mm2"] / np.pi
)
bio["radius_source"] = np.where(
    derived, "derived_from_area", "plugin_measurement"
)

bio = bio.sort_values(["strain_id","replicate_id","date"]).reset_index(drop=True)
bio["t_days"] = bio.groupby(["strain_id","replicate_id"])["date"].transform(
    lambda s: (s-s.min()).dt.total_seconds()/86400
)

print("Biological date-level observations:", len(bio))
display(bio.head())


In [ ]:
# REPLICATE-LEVEL GROWTH PARAMETERS

def slope_summary(g, variable):
    x = pd.to_numeric(g["t_days"], errors="coerce")
    y = pd.to_numeric(g[variable], errors="coerce")
    ok = x.notna() & y.notna() & np.isfinite(x) & np.isfinite(y)
    x, y = x[ok].to_numpy(), y[ok].to_numpy()

    if len(x) < MIN_POINTS_FOR_RATE or len(np.unique(x)) < 2:
        return np.nan, np.nan

    lr = stats.linregress(x,y)
    return float(lr.slope), float(lr.rvalue**2)

def summarize_replicate(g, strain_id, replicate_id):
    g = g.sort_values("t_days")
    out = {
        "strain_id": strain_id,
        "replicate_id": replicate_id,
        "n_observations": len(g),
        "duration_days": float(g["t_days"].max())
    }

    if "area_mm2" in g.columns:
        a = pd.to_numeric(g["area_mm2"], errors="coerce")
        out["initial_area_mm2"] = a.iloc[0]
        out["final_area_mm2"] = a.iloc[-1]
        out["max_area_mm2"] = a.max()
        out["area_rate_mm2_day"], out["area_rate_r2"] = slope_summary(g,"area_mm2")

    r = pd.to_numeric(g["equivalent_radius_mm"], errors="coerce")
    if r.notna().any():
        out["initial_radius_mm"] = r.iloc[0]
        out["final_radius_mm"] = r.iloc[-1]
        out["max_radius_mm"] = r.max()
        out["radial_rate_mm_day"], out["radial_rate_r2"] = slope_summary(
            g,"equivalent_radius_mm"
        )

    for v in ["circularity","aspect_ratio","eccentricity","solidity","convexity","perimeter_mm"]:
        if v in g.columns:
            z = pd.to_numeric(g[v], errors="coerce")
            if z.notna().any():
                out[f"final_{v}"] = z.iloc[-1]
                out[f"mean_{v}"] = z.mean()

    return pd.Series(out)

summary_rows = []
for (strain_id, replicate_id), g in bio.groupby(["strain_id", "replicate_id"], sort=True):
    summary_rows.append(summarize_replicate(g, strain_id, replicate_id))

replicate_summary = pd.DataFrame(summary_rows)

replicate_summary.to_csv(TABLE_DIR/"replicate_growth_parameters.csv", index=False)
display(replicate_summary.round(3))


In [ ]:
# BUILD CENTRAL, SATELLITE, AND GLOBAL COLONY OBSERVATIONS

def norm_label(x):
    return re.sub(r"[^A-Za-z0-9]+", "_", str(x).upper()).strip("_")

label_source = "final_type" if "final_type" in selected.columns else ("proposed_type" if "proposed_type" in selected.columns else None)
if label_source is None:
    raise ValueError("No final_type/proposed_type column is available to separate central and satellite colonies.")
selected["_colony_label_norm"] = selected[label_source].map(norm_label)
selected["_is_satellite"] = selected["_colony_label_norm"].isin(SATELLITE_LABELS)
selected["_is_central"] = selected["_main_candidate"]
if "area_mm2" in selected.columns:
    selected["area_mm2"] = pd.to_numeric(selected["area_mm2"], errors="coerce")

# Image-level composition. Satellite count means individually distinguishable SAT objects.
rows=[]
for key,g in selected.groupby(["strain_id","replicate_id","date","_image_key"],sort=False):
    strain_id,replicate_id,date,image_key=key
    cen=g.loc[g["_is_central"],"area_mm2"].dropna()
    sat=g.loc[g["_is_satellite"],"area_mm2"].dropna()
    rows.append({"strain_id":strain_id,"replicate_id":replicate_id,"date":date,"_image_key":image_key,
                 "central_area_mm2":cen.sum() if len(cen) else np.nan,
                 "satellite_total_area_mm2":sat.sum() if len(sat) else 0.0,
                 "n_satellites_identifiable":int(len(sat))})
composition_img=pd.DataFrame(rows)
composition_img["global_area_mm2"]=composition_img["central_area_mm2"].fillna(0)+composition_img["satellite_total_area_mm2"].fillna(0)
composition_img["satellite_fraction_global"]=np.where(composition_img["global_area_mm2"]>0,composition_img["satellite_total_area_mm2"]/composition_img["global_area_mm2"],np.nan)
composition_img["mean_identifiable_satellite_area_mm2"]=np.where(composition_img["n_satellites_identifiable"]>0,composition_img["satellite_total_area_mm2"]/composition_img["n_satellites_identifiable"],np.nan)

composition=(composition_img.groupby(["strain_id","replicate_id","date"],as_index=False)
 .agg({"central_area_mm2":"median","satellite_total_area_mm2":"median","global_area_mm2":"median",
       "n_satellites_identifiable":"median","satellite_fraction_global":"median",
       "mean_identifiable_satellite_area_mm2":"median"})
 .sort_values(["strain_id","replicate_id","date"]).reset_index(drop=True))
composition["n_satellites_identifiable"]=composition["n_satellites_identifiable"].round().astype(int)
composition["t_days"]=composition.groupby(["strain_id","replicate_id"])["date"].transform(lambda s:(s-s.min()).dt.total_seconds()/86400)

satellite_rep_rows=[]
for (strain_id,replicate_id),g in composition.groupby(["strain_id","replicate_id"],sort=True):
    g=g.sort_values("t_days"); positive=g["n_satellites_identifiable"]>0
    first_day=float(g.loc[g.index[positive][0],"t_days"]) if positive.any() else np.nan
    final=g.iloc[-1]; sat_area=pd.to_numeric(g["satellite_total_area_mm2"],errors="coerce"); sat_count=pd.to_numeric(g["n_satellites_identifiable"],errors="coerce"); sat_frac=pd.to_numeric(g["satellite_fraction_global"],errors="coerce")
    row={"strain_id":strain_id,"replicate_id":replicate_id,"has_identifiable_satellites":bool(positive.any()),
         "days_to_first_identifiable_satellite":first_day,"max_identifiable_satellite_count":sat_count.max(),
         "final_identifiable_satellite_count":sat_count.iloc[-1],"final_satellite_area_mm2":sat_area.iloc[-1],
         "max_satellite_area_mm2":sat_area.max(),"mean_satellite_area_mm2_when_identifiable":(sat_area[sat_count>0]/sat_count[sat_count>0]).mean() if (sat_count>0).any() else np.nan,
         "final_satellite_fraction_global":sat_frac.iloc[-1],"max_satellite_fraction_global":sat_frac.max(),
         "final_central_area_mm2":final["central_area_mm2"],"final_global_area_mm2":final["global_area_mm2"]}
    if len(g)>=3:
        ns,_=slope_summary(g,"n_satellites_identifiable"); aa,_=slope_summary(g,"satellite_total_area_mm2")
        row["satellite_count_rate"]=ns; row["satellite_area_rate_mm2_day"]=aa
        row["count_down_area_up_pattern"]=bool(np.isfinite(ns) and np.isfinite(aa) and ns<0 and aa>0)
    else:
        row["satellite_count_rate"]=np.nan; row["satellite_area_rate_mm2_day"]=np.nan; row["count_down_area_up_pattern"]=False
    satellite_rep_rows.append(row)
satellite_summary=pd.DataFrame(satellite_rep_rows)

global_rows=[]
for (strain_id,replicate_id),g in composition.groupby(["strain_id","replicate_id"],sort=True):
    g=g.sort_values("t_days"); row={"strain_id":strain_id,"replicate_id":replicate_id,"n_observations":len(g),"duration_days":float(g["t_days"].max())}
    for v in ["central_area_mm2","satellite_total_area_mm2","global_area_mm2"]:
        z=pd.to_numeric(g[v],errors="coerce"); row[f"initial_{v}"]=z.iloc[0]; row[f"final_{v}"]=z.iloc[-1]; row[f"max_{v}"]=z.max(); row[f"{v}_rate"],row[f"{v}_rate_r2"]=slope_summary(g,v)
    global_rows.append(row)
global_replicate_summary=pd.DataFrame(global_rows)

composition.to_csv(TABLE_DIR/"central_satellite_global_date_observations.csv",index=False)
satellite_summary.to_csv(TABLE_DIR/"satellite_replicate_summary.csv",index=False)
global_replicate_summary.to_csv(TABLE_DIR/"global_replicate_growth_parameters.csv",index=False)
print("Central/satellite/global date-level observations:",len(composition))
print("Replicates with at least one identifiable satellite:",int(satellite_summary["has_identifiable_satellites"].sum()),"of",len(satellite_summary))
display(satellite_summary.round(3))


In [ ]:
# CENTRAL, SATELLITE, AND GLOBAL TRAJECTORIES

def plot_composition_curves(df, variables, labels, filename, title):
    avail=[v for v in variables if v in df.columns]
    if not avail:return None
    fig,ax=plt.subplots(figsize=(10,5.5))
    for v,label in zip(avail,labels):
        q=df.groupby("t_days")[v].agg(["mean","std"]).reset_index().sort_values("t_days")
        ax.plot(q["t_days"],q["mean"],marker="o",label=label)
        sd=q["std"].fillna(0)
        ax.fill_between(q["t_days"],q["mean"]-sd,q["mean"]+sd,alpha=.12)
    ax.set_xlabel("Days since first observation of replicate")
    ax.set_ylabel("Area (mm²)")
    ax.set_title(title)
    ax.grid(alpha=.22)
    ax.legend()
    fig.tight_layout()
    path=FIG_DIR/filename
    fig.savefig(path,dpi=180,bbox_inches="tight")
    plt.show()
    plt.close(fig)
    return path

fig_composition=plot_composition_curves(
    composition,
    ["central_area_mm2","satellite_total_area_mm2","global_area_mm2"],
    ["Central colony","Satellites (identifiable total)","Global colony"],
    "central_satellite_global_area.png",
    "Central, satellite, and global colony area"
)

fig,ax=plt.subplots(figsize=(10,5))
q=composition.groupby("t_days")["n_satellites_identifiable"].agg(["mean","std"]).reset_index().sort_values("t_days")
ax.plot(q["t_days"],q["mean"],marker="o")
sd=q["std"].fillna(0)
ax.fill_between(q["t_days"],q["mean"]-sd,q["mean"]+sd,alpha=.12)
ax.set_xlabel("Days since first observation of replicate")
ax.set_ylabel("Number of individually distinguishable satellites")
ax.set_title("Identifiable satellite count through time")
ax.grid(alpha=.22)
fig.tight_layout()
fig_sat_count=FIG_DIR/"satellite_count_trajectory.png"
fig.savefig(fig_sat_count,dpi=180,bbox_inches="tight")
plt.show()
plt.close(fig)

# FINAL SATELLITE CONTRIBUTION — show every biological replicate as a point
# in addition to the boxplot, so strains with one or two replicates remain
# interpretable. A replicate is included if it had at least one identifiable
# satellite at any observation; the plotted value is the satellite fraction
# at that replicate's final observation.
fig,ax=plt.subplots(figsize=(12,5.5))
vals=[]
labels=[]
point_groups=[]
for strain,g in satellite_summary.groupby("strain_id"):
    z=g.loc[
        g["has_identifiable_satellites"],
        "final_satellite_fraction_global"
    ].dropna()
    if len(z):
        vals.append(z.to_numpy())
        labels.append(strain)
        point_groups.append(z.to_numpy())

if vals:
    positions=np.arange(1,len(vals)+1)
    ax.boxplot(vals,positions=positions,widths=0.55)
    rng=np.random.default_rng(42)
    for x,y in zip(positions,point_groups):
        jitter=rng.uniform(-0.10,0.10,size=len(y))
        ax.scatter(np.full(len(y),x)+jitter,y,zorder=3,alpha=.85)
        if len(y)==1:
            ax.annotate(
                "n=1",
                (x,y[0]),
                xytext=(0,8),
                textcoords="offset points",
                ha="center",
                fontsize=8
            )
    ax.set_xticks(positions)
    ax.set_xticklabels([
        f"{s}\n(n={len(y)})"
        for s,y in zip(labels,point_groups)
    ])
    ax.set_ylabel("Satellite area / total colony area")
    ax.set_xlabel("Strain (n = satellite-positive biological replicates)")
    ax.set_title("Satellite contribution at final observation")
    ax.set_ylim(bottom=0)
    ax.grid(alpha=.2,axis="y")
    ax.text(
        0.0,-0.27,
        "Each point is one biological replicate. Included replicates had at least one\n"
        "identifiable satellite at some observation; the y-value is the satellite fraction\n"
        "at the final observation. A final value near 0 can therefore occur even in a\n"
        "replicate that was satellite-positive earlier in the experiment.",
        transform=ax.transAxes,
        fontsize=8.5,
        color="dimgray",
        va="top"
    )
    fig.tight_layout()
    fig.subplots_adjust(bottom=.28)
    fig_sat_fraction=FIG_DIR/"satellite_fraction_final.png"
    fig.savefig(fig_sat_fraction,dpi=180,bbox_inches="tight")
    plt.show()
    plt.close(fig)
else:
    fig_sat_fraction=None


In [ ]:
# GROWTH CURVES

def plot_strain_curves(df, variable, ylabel, filename):
    if variable not in df.columns:
        return None
    d = df.dropna(subset=[variable,"t_days"]).copy()
    if d.empty:
        return None

    fig, ax = plt.subplots(figsize=(10,5))
    for strain, g in d.groupby("strain_id"):
        q = (
            g.groupby("t_days")[variable]
             .agg(["mean","std"])
             .reset_index()
             .sort_values("t_days")
        )
        ax.plot(q["t_days"],q["mean"],marker="o",label=strain)
        sd = q["std"].fillna(0)
        ax.fill_between(q["t_days"],q["mean"]-sd,q["mean"]+sd,alpha=.15)

    ax.set_xlabel("Days since first observation of replicate")
    ax.set_ylabel(ylabel)
    ax.set_title(f"{ylabel} — strain trajectories (mean ± SD)")
    ax.grid(alpha=.25)
    ax.legend(bbox_to_anchor=(1.02,1),loc="upper left",fontsize=8)
    fig.tight_layout()
    path = FIG_DIR/filename
    fig.savefig(path,dpi=180,bbox_inches="tight")
    plt.show()
    plt.close(fig)
    return path

def plot_replicate_curves(df, variable, ylabel, filename):
    if variable not in df.columns:
        return None
    d = df.dropna(subset=[variable,"t_days"]).copy()
    if d.empty:
        return None

    fig, ax = plt.subplots(figsize=(11,6))
    for (strain,rep), g in d.groupby(["strain_id","replicate_id"]):
        g = g.sort_values("t_days")
        ax.plot(g["t_days"],g[variable],marker="o",alpha=.75,
                label=f"{strain} — {rep}")

    ax.set_xlabel("Days since first observation of replicate")
    ax.set_ylabel(ylabel)
    ax.set_title(f"{ylabel} — individual replicate trajectories")
    ax.grid(alpha=.25)
    ax.legend(bbox_to_anchor=(1.02,1),loc="upper left",fontsize=7)
    fig.tight_layout()
    path = FIG_DIR/filename
    fig.savefig(path,dpi=180,bbox_inches="tight")
    plt.show()
    plt.close(fig)
    return path

fig_area = plot_strain_curves(
    bio,"area_mm2","Colony area (mm²)","growth_area_strain_mean.png"
)
fig_radius = plot_replicate_curves(
    bio,"equivalent_radius_mm","Equivalent radius (mm)",
    "growth_radius_replicates.png"
)


In [ ]:
# MORPHOLOGY

morphology_vars = ["circularity","aspect_ratio","eccentricity","solidity","convexity"]
available_morphology = [v for v in morphology_vars if v in bio.columns]
morphology_summary = pd.DataFrame()

for v in available_morphology:
    fig, ax = plt.subplots(figsize=(10,5))
    data, labels = [], []
    for strain, g in bio.groupby("strain_id"):
        vals = pd.to_numeric(g[v],errors="coerce").dropna().to_numpy()
        if len(vals):
            data.append(vals)
            labels.append(strain)

    if data:
        ax.boxplot(data,labels=labels)
        ax.set_ylabel(v)
        ax.set_title(f"{v} by strain")
        ax.tick_params(axis="x",rotation=60)
        ax.grid(alpha=.2,axis="y")
        fig.tight_layout()
        fig.savefig(FIG_DIR/f"morphology_{slug(v)}.png",dpi=180,bbox_inches="tight")
        plt.show()
        plt.close(fig)

if available_morphology:
    rows = []
    for strain,g in bio.groupby("strain_id"):
        row = {"strain_id":strain}
        for v in available_morphology:
            z = pd.to_numeric(g[v],errors="coerce").dropna()
            row[f"{v}_mean"] = z.mean() if len(z) else np.nan
            row[f"{v}_sd"] = z.std() if len(z)>1 else np.nan
        rows.append(row)
    morphology_summary = pd.DataFrame(rows)
    morphology_summary.to_csv(TABLE_DIR/"strain_morphology_summary.csv",index=False)

display(morphology_summary.round(3))


In [ ]:
# STRAIN-LEVEL OMNIBUS STATISTICS
# The replicate is the experimental unit.

ENDPOINTS = [
    "final_area_mm2","max_area_mm2","area_rate_mm2_day",
    "final_radius_mm","max_radius_mm","radial_rate_mm_day"
]

def eta_squared(groups):
    allvals = np.concatenate(groups)
    grand = allvals.mean()
    ss_between = sum(len(x)*(x.mean()-grand)**2 for x in groups)
    ss_total = sum(((x-grand)**2).sum() for x in groups)
    return ss_between/ss_total if ss_total else np.nan

rows = []
for endpoint in ENDPOINTS:
    if endpoint not in replicate_summary.columns:
        continue

    d = replicate_summary[["strain_id",endpoint]].copy()
    d[endpoint] = pd.to_numeric(d[endpoint],errors="coerce")
    d = d.dropna()

    groups = [g[endpoint].to_numpy() for _,g in d.groupby("strain_id") if len(g)>=2]
    if len(groups)<2:
        continue

    row = {"endpoint":endpoint,"n_replicates":len(d)}
    try:
        a = stats.f_oneway(*groups)
        row.update(anova_F=a.statistic,anova_p=a.pvalue,
                   eta_squared=eta_squared(groups))
    except Exception:
        row.update(anova_F=np.nan,anova_p=np.nan,eta_squared=np.nan)

    try:
        k = stats.kruskal(*groups)
        row.update(kruskal_H=k.statistic,kruskal_p=k.pvalue)
    except Exception:
        row.update(kruskal_H=np.nan,kruskal_p=np.nan)

    rows.append(row)

omnibus = pd.DataFrame(rows)
omnibus.to_csv(TABLE_DIR/"strain_omnibus_statistics.csv",index=False)
display(omnibus.round(4))


In [ ]:
# EXPLORATORY PAIRWISE WELCH TESTS + HOLM CORRECTION

rows = []
for endpoint in ENDPOINTS:
    if endpoint not in replicate_summary.columns:
        continue

    d = replicate_summary[["strain_id",endpoint]].copy()
    d[endpoint] = pd.to_numeric(d[endpoint],errors="coerce")
    d = d.dropna()
    strains = sorted(d["strain_id"].unique())

    for i,s1 in enumerate(strains):
        for s2 in strains[i+1:]:
            a = d.loc[d["strain_id"]==s1,endpoint].to_numpy()
            b = d.loc[d["strain_id"]==s2,endpoint].to_numpy()
            if len(a)<2 or len(b)<2:
                continue
            t = stats.ttest_ind(a,b,equal_var=False,nan_policy="omit")
            rows.append({
                "endpoint":endpoint,"strain_1":s1,"strain_2":s2,
                "n_1":len(a),"n_2":len(b),"welch_t":t.statistic,
                "p_raw":t.pvalue
            })

pairwise = pd.DataFrame(rows)

if not pairwise.empty:
    pairwise["p_holm"] = np.nan
    for endpoint,idx in pairwise.groupby("endpoint").groups.items():
        idx = list(idx)
        p = pairwise.loc[idx,"p_raw"].to_numpy()
        order = np.argsort(p)
        adj = np.empty(len(p))
        running = 0
        m = len(p)
        for rank,pos in enumerate(order):
            running = max(running,(m-rank)*p[pos])
            adj[pos] = min(running,1.0)
        pairwise.loc[idx,"p_holm"] = adj

    pairwise.to_csv(TABLE_DIR/"pairwise_welch_holm.csv",index=False)

display(pairwise.round(4))


In [ ]:
# PCA + HIERARCHICAL CLUSTERING

profile_candidates = [
    "area_rate_mm2_day","radial_rate_mm_day","final_area_mm2","final_radius_mm",
    "mean_circularity","mean_aspect_ratio","mean_eccentricity",
    "mean_solidity","mean_convexity",
    "final_global_area_mm2","global_area_mm2_rate","final_satellite_area_mm2","final_satellite_fraction_global"
]
profile_vars = [v for v in profile_candidates if v in replicate_summary.columns]

if len(profile_vars)>=2 and replicate_summary["strain_id"].nunique()>=3:
    profile = replicate_summary.groupby("strain_id")[profile_vars].mean()
    profile = profile.loc[:,profile.notna().sum()>=max(3,int(.7*len(profile)))]

    if profile.shape[0]>=3 and profile.shape[1]>=2:
        X = profile.fillna(profile.median(numeric_only=True))
        Xz = StandardScaler().fit_transform(X)

        pca = PCA(n_components=2)
        scores = pca.fit_transform(Xz)
        pca_df = pd.DataFrame(scores,index=X.index,columns=["PC1","PC2"])

        fig,ax = plt.subplots(figsize=(8,6))
        ax.scatter(pca_df["PC1"],pca_df["PC2"])
        for label,row in pca_df.iterrows():
            ax.annotate(str(label),(row["PC1"],row["PC2"]),
                        xytext=(5,5),textcoords="offset points")
        ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
        ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
        ax.set_title("Phenotypic similarity of strains — PCA")
        ax.grid(alpha=.2)
        fig.tight_layout()
        fig.savefig(FIG_DIR/"pca_strain_similarity.png",dpi=180,bbox_inches="tight")
        plt.show()
        plt.close(fig)

        Z = linkage(Xz,method="ward")
        fig,ax = plt.subplots(figsize=(10,6))
        dendrogram(Z,labels=profile.index.tolist(),ax=ax)
        ax.set_title("Hierarchical clustering of strain phenotypic profiles")
        ax.set_ylabel("Ward distance")
        ax.tick_params(axis="x",rotation=60)
        fig.tight_layout()
        fig.savefig(FIG_DIR/"strain_dendrogram.png",dpi=180,bbox_inches="tight")
        plt.show()
        plt.close(fig)

        profile.to_csv(TABLE_DIR/"strain_phenotypic_profile.csv")
        print("PCA/clustering completed.")
    else:
        print("PCA/clustering skipped: insufficient complete profiles.")
else:
    print("PCA/clustering skipped: insufficient strains or variables.")


In [ ]:
# AUTOMATIC BIOLOGICAL CONCLUSIONS
conclusions=[]
n_strains=replicate_summary["strain_id"].nunique(); n_reps=len(replicate_summary)
conclusions.append(f"The analysis contains {n_strains} strain(s) and {n_reps} biological replicate(s).")
rep_distribution=replicate_summary.groupby("strain_id").size()
conclusions.append("The number of replicates is consistent across strains." if rep_distribution.nunique()==1 else "Replicate numbers differ among strains; sample sizes are reported explicitly.")
for endpoint,label in [("radial_rate_mm_day","radial expansion rate"),("area_rate_mm2_day","area expansion rate"),("final_area_mm2","final central colony area")]:
    if endpoint in replicate_summary.columns:
        d=replicate_summary[["strain_id",endpoint]].dropna(); means=d.groupby("strain_id")[endpoint].mean()
        if len(means)>=2:
            hi,lo=means.idxmax(),means.idxmin(); conclusions.append(f"Descriptively, {label} varied among strains; {hi} had the highest observed mean and {lo} the lowest. This is descriptive and is not by itself evidence of a statistically supported difference.")
if not omnibus.empty and ((omnibus["anova_p"]<ALPHA)|(omnibus["kruskal_p"]<ALPHA)).any(): conclusions.append(f"At least one tested endpoint had an omnibus p-value below {ALPHA:g}; pairwise results and replicate variability should be inspected.")
elif not omnibus.empty: conclusions.append(f"No tested endpoint had an omnibus p-value below {ALPHA:g}.")
if not satellite_summary.empty:
    n_sat=int(satellite_summary["has_identifiable_satellites"].sum()); n_total=len(satellite_summary); conclusions.append(f"Identifiable satellites were recorded in {n_sat} of {n_total} biological replicates; the remainder had no individually distinguishable satellite colony in the analyzed observations.")
    first_days=satellite_summary["days_to_first_identifiable_satellite"].dropna()
    if len(first_days): conclusions.append(f"Among satellite-positive replicates, the first individually distinguishable satellite appeared after a median of {first_days.median():.1f} days (range {first_days.min():.1f}–{first_days.max():.1f}).")
    pattern_n=int(satellite_summary["count_down_area_up_pattern"].sum())
    if pattern_n: conclusions.append(f"{pattern_n} replicate(s) showed decreasing individually distinguishable satellite count while total satellite area increased; this is compatible with colony expansion plus merging or loss of separability and is not treated as demonstrated biological disappearance.")
conclusions.append("Repeated time points within a replicate were not treated as independent biological replicates.")
conclusions.append("Phenotypic clustering describes measured similarity and is not a taxonomic classification.")
for i,c in enumerate(conclusions,1): print(f"{i}. {c}")


In [ ]:
# CREATE STANDALONE HTML REPORT — POLISHED VERSION

def image_uri(path):
    path=Path(path)
    if not path.exists(): return None
    return "data:image/png;base64,"+base64.b64encode(path.read_bytes()).decode("ascii")
def table_html(df):
    if df is None or df.empty:return "<p class='muted'><em>No data available.</em></p>"
    x=df.copy()
    for c in x.select_dtypes(include=[np.number]).columns:x[c]=x[c].round(4)
    return x.to_html(index=False,classes="dataframe",border=0,escape=True)
def metric(label,value): return f"<div class='metric'><div class='metric-value'>{escape(str(value))}</div><div class='metric-label'>{escape(label)}</div></div>"
css="""
:root{--ink:#24302b;--muted:#68756f;--paper:#f7f5ef;--card:#fffdf8;--line:#dfe3dc;--accent:#2f6f5e;--accent2:#a46f43;--soft:#eaf1ed;--warn:#f5eadc}*{box-sizing:border-box}html{scroll-behavior:smooth}body{margin:0;background:var(--paper);color:var(--ink);font-family:Inter,ui-sans-serif,system-ui,-apple-system,BlinkMacSystemFont,"Segoe UI",sans-serif;line-height:1.6}.container{max-width:1180px;margin:0 auto;padding:0 28px}.hero{padding:68px 0 54px;background:linear-gradient(135deg,#eaf1ed,#f7f5ef 58%,#efe5d8);border-bottom:1px solid var(--line)}.eyebrow{text-transform:uppercase;letter-spacing:.16em;font-size:.75rem;font-weight:800;color:var(--accent);margin-bottom:12px}.hero h1{font-family:Georgia,serif;font-size:clamp(2.2rem,5vw,4.2rem);line-height:1.05;margin:0 0 18px;max-width:900px}.hero p{max-width:820px;color:#52615a;font-size:1.05rem}.meta{display:flex;gap:12px;flex-wrap:wrap;margin-top:22px}.pill{border:1px solid rgba(47,111,94,.2);background:rgba(255,255,255,.65);padding:7px 11px;border-radius:999px;font-size:.82rem}nav{position:sticky;top:0;z-index:10;background:rgba(247,245,239,.95);backdrop-filter:blur(10px);border-bottom:1px solid var(--line)}nav .container{display:flex;gap:18px;overflow:auto;padding-top:11px;padding-bottom:11px}nav a{color:var(--muted);text-decoration:none;white-space:nowrap;font-size:.83rem;font-weight:700}nav a:hover{color:var(--accent)}main{padding:34px 0 80px}section{scroll-margin-top:70px;margin:44px 0 0}.section-kicker{text-transform:uppercase;letter-spacing:.14em;color:var(--accent);font-size:.72rem;font-weight:800}.section-title{font-family:Georgia,serif;font-size:2rem;margin:4px 0 18px}.lead{font-size:1.02rem;color:#52615a;max-width:860px}.metrics{display:grid;grid-template-columns:repeat(auto-fit,minmax(175px,1fr));gap:14px;margin:24px 0}.metric,.card,.figure{background:var(--card);border:1px solid var(--line);border-radius:18px;box-shadow:0 7px 22px rgba(36,48,43,.045)}.metric{padding:19px 18px}.metric-value{font-size:1.75rem;font-weight:800;color:var(--accent)}.metric-label{font-size:.82rem;font-weight:800;margin-top:2px}.card{padding:22px;margin:18px 0}.callout{border-left:5px solid var(--accent);background:var(--soft);padding:18px 20px;border-radius:0 14px 14px 0;margin:20px 0}.warning{border-left-color:var(--accent2);background:var(--warn)}.muted{color:var(--muted)}.figure{padding:18px;margin:20px 0}.figure h3{font-family:Georgia,serif;margin:0 0 12px}.figure img{display:block;max-width:100%;height:auto;margin:auto;border-radius:10px}.caption{font-size:.82rem;color:var(--muted);margin-top:10px}.table-wrap{overflow:auto;border:1px solid var(--line);border-radius:14px;background:var(--card);margin:16px 0}.dataframe{border-collapse:collapse;width:100%;font-size:.82rem}.dataframe th,.dataframe td{padding:9px 10px;border-bottom:1px solid var(--line);text-align:left;white-space:nowrap}.dataframe th{background:#eef2ee;color:#30443b}.dataframe tbody tr:nth-child(even){background:#fbfaf6}.grid2{display:grid;grid-template-columns:repeat(auto-fit,minmax(320px,1fr));gap:18px}.footer{border-top:1px solid var(--line);padding:30px 0 55px;color:var(--muted);font-size:.8rem}ul.clean{padding-left:20px}li{margin:7px 0}@media(max-width:700px){.container{padding:0 17px}.hero{padding:44px 0}.section-title{font-size:1.65rem}.metrics{grid-template-columns:repeat(2,1fr)}.metric-value{font-size:1.4rem}}
"""
html=["<!doctype html><html lang='en'><head><meta charset='utf-8'><meta name='viewport' content='width=device-width,initial-scale=1'><title>FungiGrowthJ — Biological Analysis Report</title><style>"+css+"</style></head><body>"]
html.append(f"<header class='hero'><div class='container'><div class='eyebrow'>FungiGrowthJ · Biological analysis</div><h1>Growth, morphology & colony architecture</h1><p>Automated analysis of the raw FungiGrowthJ results archive, with explicit separation of the central colony, individually distinguishable satellites, and the global colony phenotype.</p><div class='meta'><span class='pill'>Input: {escape(ZIP_PATH.name)}</span><span class='pill'>Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}</span><span class='pill'>Main label: {escape(str(main_label))}</span></div></div></header>")
html.append("<nav><div class='container'><a href='#overview'>Overview</a><a href='#quality'>Quality control</a><a href='#central'>Central colony</a><a href='#satellites'>Satellites</a><a href='#global'>Global phenotype</a><a href='#morphology'>Morphology</a><a href='#stats'>Statistics</a><a href='#multivariate'>PCA & clustering</a><a href='#conclusions'>Conclusions</a></div></nav><main class='container'>")
html.append("<section id='overview'><div class='section-kicker'>02 · Experiment</div><h2 class='section-title'>Data and description</h2><div class='metrics'>"+metric('Strains',n_strains)+metric('Biological replicates',n_reps)+metric('Date-level observations',len(bio))+metric('Images after revision resolution',selected['_image_key'].nunique())+metric('Raw result CSVs',len(result_files))+"</div><div class='callout'><strong>Experimental unit:</strong> the biological replicate. Repeated observations through time are used for trajectories and rate estimation, not counted as independent replicates for strain-level inference.</div></section>")
html.append(f"<section id='quality'><div class='section-kicker'>03 · QC</div><h2 class='section-title'>Data audit & quality control</h2><div class='card'><p>Raw rows: <strong>{len(raw)}</strong> · rows after revision resolution: <strong>{len(selected)}</strong> · images with multiple main candidates: <strong>{int((main_counts>1).sum())}</strong> · multiple-main policy: <strong>{escape(MULTIPLE_MAIN_POLICY)}</strong>.</p></div><div class='grid2'><div><h3>Replicates per strain</h3><div class='table-wrap'>{table_html(rep_counts)}</div></div><div><h3>Dates per biological replicate</h3><div class='table-wrap'>{table_html(date_counts)}</div></div></div></section>")

def figblock(title,path,caption):
    if path is None or not Path(path).exists(): return ""
    return f"<div class='figure'><h3>{escape(title)}</h3><img src='{image_uri(path)}'><div class='caption'>{escape(caption)}</div></div>"
html.append("<section id='central'><div class='section-kicker'>04 · Central colony</div><h2 class='section-title'>Central colony growth</h2><p class='lead'>The central colony is identified explicitly from the plugin classification rather than inferred from size. Growth rates are observed average slopes over the sampled interval.</p>"+figblock('Strain trajectories',fig_area,'Mean ± SD across biological replicates.')+figblock('Individual replicate trajectories',fig_radius,'Individual temporal trajectories illustrate within-strain replicate variability.')+"<div class='table-wrap'>"+table_html(replicate_summary)+"</div></section>")

sat_note="<div class='callout warning'><strong>Interpretation rule:</strong> if distinguishable satellite count decreases while total satellite area increases, the pattern is treated as compatible with colony expansion, merging, or loss of separability. It is not interpreted as demonstrated biological disappearance.</div>"
nsat=int(satellite_summary['has_identifiable_satellites'].sum()); ntot=len(satellite_summary); pos=satellite_summary.loc[satellite_summary['has_identifiable_satellites'],'days_to_first_identifiable_satellite'].dropna()
html.append("<section id='satellites'><div class='section-kicker'>05 · Colony architecture</div><h2 class='section-title'>Satellite colonies</h2><p class='lead'>Satellite measurements describe individually distinguishable colonies. A satellite count is therefore a measure of separability, not necessarily a direct count of persistent biological entities.</p>"+sat_note+"<div class='metrics'>"+metric('Replicates with satellites',f'{nsat}/{ntot}')+metric('Without identifiable satellites',ntot-nsat)+metric('Median days to first satellite',f'{pos.median():.1f}' if len(pos) else '—')+metric('Count↓ + area↑ patterns',int(satellite_summary['count_down_area_up_pattern'].sum()))+"</div><div class='callout'><strong>How to read the final satellite fraction:</strong> points represent biological replicates, not individual images. A replicate is included when it had at least one individually distinguishable satellite at any observation; its plotted value is the satellite fraction at the final observation. Therefore a value near 0 can occur after satellites were present earlier in the experiment.</div>"+figblock('Central, satellite and global area',fig_composition,'Satellite area is the sum of individually distinguishable SAT objects; global area is central + satellite area.')+figblock('Identifiable satellite count through time',fig_sat_count,'Tracks individually distinguishable satellites; decreases are not automatically interpreted as biological disappearance.')+figblock('Satellite contribution at final observation',fig_sat_fraction,'Each point is one biological replicate. Replicates are included if they had at least one identifiable satellite at any observation; the y-value is the satellite area divided by total colony area at the final observation. A low or zero final fraction does not imply the replicate was never satellite-positive.')+"<div class='table-wrap'>"+table_html(satellite_summary)+"</div></section>")
html.append("<section id='global'><div class='section-kicker'>06 · Global phenotype</div><h2 class='section-title'>Central + satellites as a whole</h2><p class='lead'>The global phenotype combines the measured central colony and the area of all individually distinguishable satellite colonies.</p><div class='table-wrap'>"+table_html(global_replicate_summary)+"</div></section>")
html.append("<section id='morphology'><div class='section-kicker'>07 · Morphology</div><h2 class='section-title'>Colony morphology</h2><p class='lead'>Morphological descriptors are treated as measured phenotypes; low circularity is not automatically considered biological invalidity.</p>"+''.join(figblock(v,FIG_DIR/f'morphology_{slug(v)}.png',f'Distribution of {v} by strain.') for v in available_morphology)+"<div class='table-wrap'>"+table_html(morphology_summary)+"</div></section>")
html.append("<section id='stats'><div class='section-kicker'>08 · Inference</div><h2 class='section-title'>Statistical comparison among strains</h2><div class='callout'><strong>Inference unit:</strong> replicate-level endpoints. Pairwise comparisons are exploratory, particularly where some strains have fewer biological replicates.</div><div class='table-wrap'>"+table_html(omnibus)+"</div><h3>Exploratory pairwise Welch tests · Holm correction</h3><div class='table-wrap'>"+table_html(pairwise)+"</div></section>")
html.append("<section id='multivariate'><div class='section-kicker'>09 · Multivariate phenotype</div><h2 class='section-title'>PCA & hierarchical clustering</h2><p class='lead'>Multivariate analyses summarize measured phenotype similarity among strains, not taxonomy or genetic relatedness.</p>"+figblock('PCA — strain phenotypic similarity',FIG_DIR/'pca_strain_similarity.png','Strain-level mean phenotype profiles after standardization.')+figblock('Hierarchical clustering',FIG_DIR/'strain_dendrogram.png','Ward clustering of standardized strain phenotype profiles.')+"</section>")
html.append("<section id='conclusions'><div class='section-kicker'>10 · Synthesis</div><h2 class='section-title'>Biological conclusions</h2><div class='card'><ul class='clean'>"+''.join(f'<li>{escape(c)}</li>' for c in conclusions)+"</ul></div></section>")
html.append("<section id='limitations'><div class='section-kicker'>11 · Reproducibility</div><h2 class='section-title'>Limitations & interpretation safeguards</h2><div class='card'><ul class='clean'><li>Biological replicate, not ROI, is the experimental unit.</li><li>Longitudinal observations are used for trajectories and slopes.</li><li>Growth rates are observed average slopes over the sampled period.</li><li>Satellite counts represent individually distinguishable objects and can change because of merging or loss of separability.</li><li>Lag phase is not inferred when temporal resolution does not support a defensible model.</li><li>Low circularity is a morphology measurement, not automatic invalidity.</li><li>Phenotypic clustering is not taxonomy.</li><li>Revision resolution is performed before biological summaries.</li></ul></div></section>")
html.append("</main><footer class='footer'><div class='container'>Generated by the generic FungiGrowthJ biological-analysis notebook · raw results ZIP as input · adaptive analysis based on available fields.</div></footer></body></html>")
report_path=OUTPUT_DIR/"FungiGrowthJ_Biological_Report.html"; report_path.write_text("\n".join(html),encoding="utf-8"); print("HTML report:",report_path.resolve())


## How to use

1. Put the FungiGrowthJ `results.zip` beside this notebook.
2. Change `ZIP_PATH` only if necessary.
3. Run **Run All**.

The notebook does not assume particular strain names, a fixed number of strains, exactly three replicates, R1/R2/R3, a fixed number of dates, or a particular FungiGrowthJ version.

It is capability-driven: measurements that are absent are skipped rather than invented.